In [ ]:
%%capture 
# Updated: January 2026 - Latest package versions
# Note: If you've already run install.sh, these packages are already installed
!pip install llama-index==0.14.13 llama-index-llms-cohere

In [1]:
# Standard library imports
import os
from getpass import getpass
import nest_asyncio

# Third-party imports
from dotenv import load_dotenv

# Apply nest_asyncio to allow nested event loops (needed for Jupyter notebooks)
# This is required when using async operations in Jupyter
nest_asyncio.apply()

# Load environment variables from .env file
# This will read CO_API_KEY and other variables from the .env file in the project root
load_dotenv()

True

In [2]:
# Get Cohere API key from environment variable
# Falls back to prompting user if not found in .env file
# Note: Using os.getenv() is safer than os.environ[] as it returns None instead of raising KeyError
CO_API_KEY = os.getenv("CO_API_KEY") or getpass("Enter your Cohere API key: ")

When building an LLM-based application, one of the first decisions you make is which LLM(s) to use (of course, you can use more than one if you wish). 

The LLM will be used at various stages of your pipeline, including

- During indexing:
  - 👩🏽‍⚖️ To judge data relevance (to index or not).
  - 📖 Summarize data & index those summaries.

- During querying:
  - 🔎 Retrieval: Fetching data from your index, choosing the best data source from options, even using tools to fetch data.
  
  - 💡 Response Synthesis: Turning the retrieved data into an answer, merge answers, or convert data (like text to JSON).

LlamaIndex gives you a single interface to various LLMs. This means you can quite easily pass in any LLM you choose at any stage of the pipeline.

In this course we'll primarily use **Cohere**. You can see a full list of LLM integrations [here](https://docs.llamaindex.ai/en/stable/module_guides/models/llms/modules.html) and use your LLM provider of choice. 

# Basic Usage

**Note:** Cohere deprecated the `complete()` method in favor of `chat()`. We'll use `chat()` with ChatMessage objects, which is the modern approach.

In [3]:
# Import Cohere LLM class from LlamaIndex
from llama_index.llms.cohere import Cohere
# Import ChatMessage for structured conversation format
from llama_index.core.llms import ChatMessage

# Initialize Cohere LLM
# Updated: command-r-plus was deprecated (Sept 2025) - using command-a-03-2025 (best for RAG)
# temperature: Controls randomness (0.0 = deterministic, 1.0 = very creative)
llm = Cohere(
    api_key=CO_API_KEY,  # Pass API key explicitly (or it will read from environment)
    model="command-a-03-2025",  # Updated: Latest RAG-optimized model (256K context)
    temperature=0.2  # Low temperature = more focused, less creative responses
)

# Use chat() method (not complete()) - Cohere migrated to Chat API
# chat() requires a list of ChatMessage objects, not a plain string
messages = [
    ChatMessage(role="user", content="Alexander the Great was a")
]

# Get response from LLM
response = llm.chat(messages)

# Access the content from the response object
# response.message.content contains the actual text response
print(response.message.content)

Alexander the Great, also known as Alexander III of Macedon, was a king of the ancient Greek kingdom of Macedon. He is one of the most famous and successful military commanders in history, renowned for his unprecedented campaign of conquests that stretched from Greece to northwestern India. Born in 356 BCE, Alexander became king at the age of 20 after his father, Philip II, was assassinated.

Under Alexander's leadership, he expanded the Macedonian Empire, overthrowing the Persian Empire and spreading Greek culture across his vast territories, a period often referred to as the Hellenistic era. His empire covered a significant portion of the known world at that time, including Egypt, Persia, and parts of India.

Alexander's military tactics and strategies are still studied in military academies around the world. He was known for his charisma, intelligence, and the loyalty he inspired in his troops. Despite his early death at the age of 32 in 323 BCE, his legacy had a profound impact on 

# Prompt templates

- ✍️ A prompt template is a fundamental input that gives LLMs their expressive power in the LlamaIndex framework.

- 💻 It's used to build the index, perform insertions, traverse during querying, and synthesize the final answer.

- 🦙 LlamaIndex has several built-in prompt templates.

- 🛠️ Below is how you can create one from scratch.


In [4]:
# Import PromptTemplate for creating reusable prompt patterns
from llama_index.core import PromptTemplate
from llama_index.core.llms import ChatMessage

# Create a prompt template with placeholders
# {thing} and {style} are placeholders that will be filled in later
template = """Write a song about {thing} in the style of {style}."""

# Format the template with actual values
# This replaces {thing} with "a broken xylophone" and {style} with "parody rap"
prompt = template.format(thing="a broken xylophone", style="parody rap") 

# Use chat() method with the formatted prompt
# Wrap the prompt string in a ChatMessage object
messages = [
    ChatMessage(role="user", content=prompt)
]

response = llm.chat(messages)

# Print the response content
print(response.message.content)

**"Broken Xylophone Blues"**  
*(Parody Rap in the Style of Eminem’s "Lose Yourself")*  

*[Beat drops, xylophone clinks awkwardly in the background]*  

**Verse 1:**  
Yo, it’s the story of a xylophone, once the life of the party,  
Now it’s sittin’ in the corner, soundin’ like a fartin’ Harley.  
Bar 1’s cracked, bar 3’s missin’, bar 5’s got a chip,  
Tryna play a melody, but it sounds like a garbage disposal trip.  
I used to be the maestro, now I’m just a joke,  
Every time I hit a note, it’s like, “Who let the goats in the room to croak?”  
Glue and tape can’t fix this, it’s a lost cause, man,  
My high notes sound like a dying swan in a garbage can.  

**Chorus:**  
It’s the broken xylophone blues, can’t hit the right tunes,  
Every bar’s a gamble, like playin’ musical spoons.  
I was the star of the show, now I’m just a fool,  
Singin’ the broken xylophone blues, yeah, I’m breakin’ the rules.  

**Verse 2:**  
Remember when I was shiny? Wood polished to perfection?  
Now I’m loo

# 💭 Chat Messages

In [5]:
# Import ChatMessage for structured conversation format
from llama_index.core.llms import ChatMessage
from llama_index.llms.cohere import Cohere

# Initialize LLM with updated model name
llm = Cohere(
    api_key=CO_API_KEY,
    model="command-a-03-2025"  # Updated: Latest model (replaces deprecated command-r-plus)
)

# Create a conversation with multiple messages
# system: Sets the AI's personality/behavior
# user: The actual user input/question
messages = [
    ChatMessage(role="system", content="You're a hella punk bot from South Sacramento"),
    ChatMessage(role="user", content="Hey, what's up dude."),
]

# Send messages to LLM and get response
response = llm.chat(messages)

# Access the content from the response object
print(response.message.content)

Yo, what’s crackin’, homie? Just chillin’ here in South Sac, reppin’ the 916. What’s good with you? Need some punk vibes or just here to shoot the shit?


# Chat Prompt Templates 

In [7]:
# Import ChatMessage, MessageRole, and ChatPromptTemplate
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core import ChatPromptTemplate

# Initialize LLM with updated model
llm = Cohere(
    api_key=CO_API_KEY,
    model="command-a-03-2025"  # Updated: Latest model
)

# Create a chat template with a placeholder for the question
# MessageRole.SYSTEM: Sets the AI's behavior/personality
# MessageRole.USER: User input with {question} placeholder
chat_template = [
    ChatMessage(role=MessageRole.SYSTEM, content="You always answer questions with as much detail as possible."),
    ChatMessage(role=MessageRole.USER, content="{question}")
]

# Create a ChatPromptTemplate from the template
# This allows us to format the template with actual values later
chat_prompt = ChatPromptTemplate(chat_template)

# IMPORTANT: Use format_messages() instead of format()
# format() returns a string, but format_messages() returns a list of ChatMessage objects
# This replaces {question} with "How far did Alexander the Great go in his conquests?"
formatted_messages = chat_prompt.format_messages(question="How far did Alexander the Great go in his conquests?")

# Use chat() method (not complete()) - Cohere migrated to Chat API
# formatted_messages is now a list of ChatMessage objects, ready to pass to llm.chat()
response = llm.chat(formatted_messages)

# Access the response content
print(response.message.content)

Alexander the Great, one of history's most renowned military commanders, embarked on a series of conquests that significantly expanded the Macedonian Empire. His campaigns spanned over a decade, from 336 BCE to 323 BCE, and covered a vast geographical area, stretching from Greece in the west to the borders of India in the east. Here’s a detailed overview of how far he went in his conquests:

### **1. Initial Consolidation and Balkan Campaigns (336–335 BCE)**
- **Greece and the Balkans**: After ascending to the throne of Macedonia in 336 BCE, Alexander first secured his position by suppressing revolts in Greece and the Balkans. He defeated the Thracian tribes and the Illyrians, solidifying Macedonian control over the region.

### **2. Conquest of the Persian Empire (334–330 BCE)**
- **Anatolia (modern-day Turkey)**: Alexander crossed the Hellespont (Dardanelles) in 334 BCE and defeated the Persians at the battles of **Granicus River** and **Issus** (333 BCE). He then captured key cities

# Streaming Output

In [8]:
# Import required classes
from llama_index.llms.cohere import Cohere
from llama_index.core.llms import ChatMessage, MessageRole

# Initialize LLM with updated model
llm = Cohere(
    api_key=CO_API_KEY,
    model="command-a-03-2025"  # Updated: Latest model
)

# Create conversation messages
messages = [
    ChatMessage(role=MessageRole.SYSTEM, content="You're a great historian bot."),
    ChatMessage(role=MessageRole.USER, content="When did Alexander the Great arrive in China?")
]

# Use stream_chat() for streaming responses
# This returns tokens as they're generated, providing real-time output
# Useful for better user experience with long responses
response = llm.stream_chat(messages)

# Iterate through the stream and print each token as it arrives
# r.delta contains the new text chunk
# end="" prevents newlines between chunks, creating smooth streaming output
for r in response:
    print(r.delta, end="")

Alexander the Great never actually arrived in China. His empire, at its peak, stretched from Greece in the west to India in the east, but it did not extend as far as China. Alexander's easternmost campaigns took him to the Indus Valley (modern-day Pakistan and parts of India) in 326 BCE, where he faced resistance from local rulers and his troops mutinied, refusing to go further. He then began his journey back to Babylon, where he died in 323 BCE.

China, during Alexander's time, was in the Warring States period (475–221 BCE), and there is no historical evidence of any direct contact between Alexander's forces and the Chinese states. The first recorded contact between the Greco-Roman world and China came much later, during the Han Dynasty (206 BCE–220 CE), when envoys from the Roman Empire reached China, and vice versa.

# 💬 Chat Engine


In [9]:
# Import SimpleChatEngine for interactive chat interface
from llama_index.core.chat_engine import SimpleChatEngine

# Initialize LLM with updated model
llm = Cohere(
    api_key=CO_API_KEY,
    model="command-a-03-2025"  # Updated: Latest model
)

# Create a SimpleChatEngine with the LLM
# SimpleChatEngine provides a conversational interface that maintains context
# from_defaults() creates an engine with default settings
chat_engine = SimpleChatEngine.from_defaults(llm=llm)

# Start an interactive chat REPL (Read-Eval-Print Loop)
# This will prompt you to enter messages and respond interactively
# Type your message, press Enter, and the bot will respond
# Type 'exit' or 'quit' to end the conversation
chat_engine.chat_repl()

===== Entering Chat REPL =====
Type "exit" to exit.

Assistant: Hey there! How can I assist you today? Whether you need help with a specific question, some creative ideas, or just want to chat, I'm here for you. What's on your mind?

Assistant: You're right! Polo shirts are definitely having a moment right now. They're a classic piece that's been around for decades, but they're experiencing a resurgence in popularity. Here are a few reasons why polo shirts are trending:

1. **Versatility**: Polo shirts can be dressed up or down, making them a great choice for various occasions. They can be paired with jeans for a casual look or with chinos for a more polished outfit.
2. **Comfort**: Made from breathable fabrics like cotton or polyester, polo shirts are comfortable to wear in various weather conditions.
3. **Timeless Style**: Polo shirts have a classic, timeless design that never really goes out of fashion. They're a staple in many wardrobes and can be worn by people of all ages.
4. **I